# Week 5 – Mathematics for Machine Learning

**Task:** Mathematical Foundations

This notebook implements selected mathematical and statistical operations using Python and NumPy. Core arithmetic is performed with explicit loops wherever practical rather than relying directly on ready-made statistical functions. NumPy is used to represent arrays and independently verify selected answers.

**Topics:** mean, median, mode, variance, standard deviation, probability, basic statistics, vectors, matrices, and matrix multiplication.

## 1. Objectives and workflow

1. Load a small dataset and inspect the numerical columns.
2. Implement descriptive statistics using loops.
3. distinguish population variance from sample variance.
4. Estimate empirical probabilities from observed counts.
5. Implement vector arithmetic and matrix operations.
6. Verify results with NumPy and interpret the outputs.

The included dataset is synthetic and used only for educational practice.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.math_operations import (
    manual_sum, manual_mean, manual_median, manual_mode, manual_min, manual_max,
    manual_variance, manual_standard_deviation, empirical_probability,
    probability_from_data, vector_add, vector_subtract, vector_scale,
    vector_dot, vector_norm, matrix_add, matrix_transpose, matrix_multiply,
    describe_numbers
)

df = pd.read_csv(PROJECT_ROOT / "data" / "student_performance.csv")
df.head()

,student_id,study_hours_per_day,exam_score,attendance_percent,assignments_completed
0,S001,2.5,78,90,8
1,S002,3.0,85,95,9
2,S003,4.0,92,98,10
3,S004,1.5,67,75,7
4,S005,3.5,85,95,9


## 2. Understand the sample data

Each row represents one synthetic student record. The numeric columns are study hours per day, exam score, attendance percentage, and assignments completed. We use exam scores as the main worked example.

In [2]:
print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))
print("Missing values by column:\n", df.isna().sum())
print("Duplicate rows:", int(df.duplicated().sum()))
scores = df["exam_score"].to_numpy(dtype=float)
study_hours = df["study_hours_per_day"].to_numpy(dtype=float)
attendance = df["attendance_percent"].to_numpy(dtype=float)
print("Exam scores:", scores)

Dataset shape: (12, 5)
Columns: ['student_id', 'study_hours_per_day', 'exam_score', 'attendance_percent', 'assignments_completed']
Missing values by column:
 student_id               0
study_hours_per_day      0
exam_score               0
attendance_percent       0
assignments_completed    0
dtype: int64
Duplicate rows: 0
Exam scores: [78. 85. 92. 67. 85. 90. 73. 88. 95. 85. 70. 82.]


## 3. Mean (arithmetic average)

Formula: \(\bar{x} = \frac{\sum_{i=1}^{n} x_i}{n}\).

The code below adds every value using a loop, then divides by the number of observations.

In [3]:
score_total = manual_sum(scores)
score_mean = manual_mean(scores)
print("Manual sum:", score_total)
print("Manual mean:", score_mean)
print("NumPy verification:", np.mean(scores))
assert np.isclose(score_mean, np.mean(scores))

Manual sum: 990.0
Manual mean: 82.5
NumPy verification: 82.5


## 4. Median and mode

**Median:** the middle value after sorting. For an even number of observations, it is the average of the two middle values.

**Mode:** the most frequently occurring value. This implementation returns all modes when there is a frequency tie.

In [4]:
from src.math_operations import manual_insertion_sort
print("Scores sorted manually:", manual_insertion_sort(scores))
print("Median:", manual_median(scores))
print("Mode(s):", manual_mode(scores))
print("NumPy median verification:", np.median(scores))
assert np.isclose(manual_median(scores), np.median(scores))

Scores sorted manually: [67.0, 70.0, 73.0, 78.0, 82.0, 85.0, 85.0, 85.0, 88.0, 90.0, 92.0, 95.0]
Median: 85.0
Mode(s): [85.0]
NumPy median verification: 85.0


## 5. Variance and standard deviation

Population variance: \(\sigma^2 = \frac{\sum (x_i-\mu)^2}{N}\).

Sample variance: \(s^2 = \frac{\sum (x_i-\bar{x})^2}{n-1}\).

Standard deviation is the square root of variance. Use population formulas when the dataset is the entire population of interest; use sample formulas when these observations are treated as a sample from a wider population.

In [5]:
pop_var = manual_variance(scores, sample=False)
sample_var = manual_variance(scores, sample=True)
pop_std = manual_standard_deviation(scores, sample=False)
sample_std = manual_standard_deviation(scores, sample=True)
print("Population variance:", pop_var)
print("Sample variance:", sample_var)
print("Population standard deviation:", pop_std)
print("Sample standard deviation:", sample_std)
assert np.isclose(pop_var, np.var(scores, ddof=0))
assert np.isclose(sample_var, np.var(scores, ddof=1))

Population variance: 71.58333333333333
Sample variance: 78.0909090909091
Population standard deviation: 8.460693430998036
Sample standard deviation: 8.836906081367454


## 6. Basic statistical summary

The summary combines count, minimum, maximum, range, mean, median, mode, variance, and standard deviation. Each is computed with the custom functions in `src/math_operations.py`.

In [6]:
score_summary = describe_numbers(scores)
pd.Series(score_summary, name="Exam score summary")

count                                   12
minimum                               67.0
maximum                               95.0
range                                 28.0
mean                                  82.5
median                                85.0
mode                                [85.0]
population_variance              71.583333
population_standard_deviation     8.460693
sample_variance                  78.090909
sample_standard_deviation         8.836906
Name: Exam score summary, dtype: object

## 7. Probability

For an empirical estimate, \(P(A) = \frac{\text{number of observed outcomes satisfying A}}{\text{total observed outcomes}}\).

Example events: exam score at least 85, and attendance at least 90%. These probabilities describe only this sample dataset.

In [7]:
p_score_85 = probability_from_data(scores, lambda x: x >= 85)
p_attendance_90 = probability_from_data(attendance, lambda x: x >= 90)
print(f"P(exam score >= 85) = {p_score_85:.3f}")
print(f"P(attendance >= 90%) = {p_attendance_90:.3f}")
print("Count-based example P(3 successes out of 8) =", empirical_probability(3, 8))

P(exam score >= 85) = 0.583
P(attendance >= 90%) = 0.667
Count-based example P(3 successes out of 8) = 0.375


## 8. Vectors

A vector is an ordered collection of values. In machine learning, a feature vector can represent one observation. The examples implement vector addition, subtraction, scalar multiplication, dot product, and Euclidean norm.

In [8]:
v1 = np.array([2.0, 3.0, 4.0])
v2 = np.array([1.0, 5.0, 2.0])
print("v1 + v2:", vector_add(v1, v2))
print("v1 - v2:", vector_subtract(v1, v2))
print("3 * v1:", vector_scale(v1, 3))
print("v1 dot v2:", vector_dot(v1, v2))
print("Norm of v1:", vector_norm(v1))
assert np.allclose(vector_add(v1, v2), v1 + v2)
assert np.isclose(vector_dot(v1, v2), np.dot(v1, v2))

v1 + v2: [3. 8. 6.]
v1 - v2: [ 1. -2.  2.]
3 * v1: [ 6.  9. 12.]
v1 dot v2: 25.0
Norm of v1: 5.385164807134504


## 9. Matrices and matrix operations

A matrix is a rectangular arrangement of values. The project implements matrix addition, transpose, and multiplication with explicit nested loops. Matrix multiplication requires the number of columns in the left matrix to equal the number of rows in the right matrix.

In [9]:
A = np.array([[1, 2, 3], [4, 5, 6]], dtype=float)
B = np.array([[7, 8], [9, 10], [11, 12]], dtype=float)
C = np.array([[5, 6, 7], [8, 9, 10]], dtype=float)
print("A + C:\n", matrix_add(A, C))
print("Transpose of A:\n", matrix_transpose(A))
product = matrix_multiply(A, B)
print("A x B:\n", product)
print("NumPy matrix product verification:\n", A @ B)
assert np.allclose(product, A @ B)

A + C:
 [[ 6.  8. 10.]
 [12. 14. 16.]]
Transpose of A:
 [[1. 4.]
 [2. 5.]
 [3. 6.]]
A x B:
 [[ 58.  64.]
 [139. 154.]]
NumPy matrix product verification:
 [[ 58.  64.]
 [139. 154.]]


## 10. Results interpretation

- The mean summarizes the arithmetic centre but can be influenced by unusually high or low scores.
- The median is less sensitive to extreme values.
- The mode shows the most frequently recorded score; multiple modes are possible.
- Standard deviation measures spread in the same units as the scores.
- Empirical probability is a proportion from observed rows and should not be treated as a universal or causal statement.
- The dot product combines corresponding vector elements and is used in linear models.
- Matrix multiplication combines feature and weight matrices in common machine-learning computations.

In [10]:
# Compact verification table
verification = pd.DataFrame({
    "Measure": ["Mean", "Median", "Population variance", "Population std. deviation"],
    "Manual implementation": [manual_mean(scores), manual_median(scores), manual_variance(scores), manual_standard_deviation(scores)],
    "NumPy reference": [np.mean(scores), np.median(scores), np.var(scores), np.std(scores)]
})
verification["Absolute difference"] = (verification["Manual implementation"] - verification["NumPy reference"]).abs()
verification

,Measure,Manual implementation,NumPy reference,Absolute difference
0,Mean,82.500000,82.500000,0.0
1,Median,85.000000,85.000000,0.0
2,Population variance,71.583333,71.583333,0.0
3,Population std. deviation,8.460693,8.460693,0.0


## 11. Conclusion

This practical demonstrates fundamental statistics, probability, vectors, and matrices needed in machine learning. The custom implementations make the formulas and iteration logic explicit. NumPy reference calculations confirm the selected statistical and matrix results. The program also validates empty inputs, non-finite values, and incompatible vector/matrix dimensions.

## 12. Further practice

1. Calculate the mean and median for attendance percentages and compare them.
2. Change the event to `exam_score >= 90` and compute the empirical probability.
3. Create two new vectors of length four and calculate their dot product.
4. Multiply a 3×2 matrix by a 2×4 matrix.
5. Explain why sample variance divides by `n - 1` rather than `n`.